In [ ]:
from classifiers.ThreeAxialTransformer import ThreeAxialTransformer

m = ThreeAxialTransformer(dims=(300,300,250), 
                          n_dim=32, 
                          n_layer=3,
                          n_class=1, 
                          pos_embed=False, 
                          pool_kernel_size=2,
                          pool_stride_size=2)

In [ ]:
import torch
device = torch.device("cuda")
print(device)

In [ ]:
m = m.to(device)

In [ ]:
print(m.get_num_params())

In [ ]:
x = [
    {"dataset": "artix", "image": r"E:\bilel\ARTIX\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"},
    {"dataset": "artix", "image": r"E:\bilel\ARTIX\ARTIX\DICOM_ARTIX_data\004\CT0\DOE^JOHN_ANON17151_CT_2017-03-30_174237_ORL.(sauf.sinus)_ORL.2MM_n211__00000"},
]

o = m(x, device)
print(o.shape)

In [1]:
import torch
from lighter_zoo import SegResEncoder

device = "cuda:0"

model = SegResEncoder.from_pretrained("project-lighter/ct_fm_feature_extractor")
model.train()
model.to(device=device)

x = torch.rand(8,1,100,100,100, device=device, dtype=torch.float32)
o = model(x)
o = torch.nn.functional.adaptive_avg_pool3d(o[-1], 1)
print(o.shape)

torch.Size([8, 512, 1, 1, 1])


In [7]:
import pandas as pd

df = pd.DataFrame({"a": list(range(10)), "b": list(range(10))})
# print(df)
print(df.mean().to_frame().T.values.shape)

(1, 2)


In [26]:
import torch

def dist(a, b, type="euclidean"):
    match type:
        case "euclidean":
            return torch.sum((a-b)**2, dim=-1)
        case _:
            return torch.sum((a-b)**2, dim=-1)

q = torch.rand(3,128)
pos_proto = torch.rand(1,128)
neg_proto = torch.rand(1,128)

all_proto = torch.stack((pos_proto, neg_proto), dim=0)
print(all_proto.shape)

all_proto = torch.repeat_interleave(all_proto, q.shape[0], dim=1)
print(all_proto.shape)

total_dist = torch.exp(-dist(q, all_proto)).sum(dim=0)
print(total_dist.shape)

print(dist(q, pos_proto).shape)

print(torch.log(torch.exp(-dist(q, pos_proto))/(total_dist + 1e-5)).shape)

print(torch.sum(-torch.log(torch.exp(-dist(q, pos_proto))/(total_dist + 1e-5))).shape)

torch.Size([2, 1, 128])
torch.Size([2, 3, 128])
torch.Size([3])
torch.Size([3])
torch.Size([3])
torch.Size([])


In [2]:
import torch
import glob

ckpt = {}
for file_ in glob.glob(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\064\accurate-auk-of-holistic-apotheosis\best_checkpoint_*.pt"):
    state_dict = torch.load(file_)
    i = int(file_.split("best_checkpoint_")[1].split(".pt")[0])
    ckpt.update({i: state_dict})

print(ckpt.keys())
torch.save(ckpt, r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\ckpt.pt")

dict_keys([0, 1, 2])


In [3]:
ckpt = torch.load(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\ckpt.pt")
print(ckpt.keys())

dict_keys([0, 1, 2])


In [1]:
import pickle
input_path = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\pickle datasets\radcure.pickle"
with open(input_path, "rb") as f:
    print("loading patients...")
    patients = pickle.load(f)
    print(f"found {len(patients)} patients in {input_path}")

loading patients...
found 3346 patients in C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\pickle datasets\radcure.pickle


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
from models import CT_CLIP
device = "cuda"
infer, preprocess, model = CT_CLIP.load(device)

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


In [4]:
import tqdm
for id_, p in tqdm.tqdm(list(patients.items()), ncols=50):
    p.sort_imaging()   # sort images to recover first CT scan (i.e., CT0)
    input_path = p.ct[0].path
    bbox = p.ct[0].get_GTV_bbox()
    output = infer(input_path, bbox, preprocess, model, device)

  0%|         | 3/3346 [01:02<19:24:50, 20.91s/it]


RuntimeError: cannot reshape tensor of 0 elements into shape [1, 0, 24, 24, -1] because the unspecified dimension size -1 can be any value and is ambiguous

In [16]:
from monai.transforms import MapTransform, Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped, ToTensord, \
    ScaleIntensityRanged, Spacingd, Orientationd, SpatialPadd, CenterSpatialCropd
from models import utils

preprocess = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    utils.BboxCropd(keys=["image"], roi_size=(480,480,10)),   # crop to GTV bounding box
])

x = preprocess({"image": p.ct[0].path, "bbox": p.ct[0].get_GTV_bbox()})["image"]
print(x.shape)

torch.Size([1, 453, 480, 10])


In [14]:
print(p.ct[0].get_GTV_bbox())

(array([299, 226,  85], dtype=int64), array([307, 234,  91], dtype=int64))


In [ ]:
    # # Preprocessing pipeline
    # preprocess = Compose([
    #     LoadImaged(keys=["image"]),
    #     EnsureChannelFirstd(keys=["image"]),
    #     utils.BboxCropd(keys=["image"], roi_size=(480,480,-1)),   # crop to GTV bounding box
    #     Spacingd(keys=["image"], pixdim=(0.75, 0.75, 1.5), mode=("bilinear")),   # resample to common spacing
    #     CropZd(keys=["image"], z_div_factor=10),   # make sure z dimension is divisible by 10 after resampling (requirement of the model)
    #     CenterSpatialCropd(keys=["image"], roi_size=(480,480,-1)),   # crop to common size (in case bbox crop results in larger size than expected, we center-crop to expected size)
    #     SpatialPadd(keys=["image"], spatial_size=(480,480,-1)),   # pad to common size (in case bbox crop results in smaller size than expected)
    #     Orientationd(keys=["image"], axcodes="SLP"),   # reorient to common orientation
    #     EnsureTyped(keys=["image"]),        
    #     ScaleIntensityRanged(
    #         keys=["image"],
    #         a_min=-1000,
    #         a_max=1000,
    #         b_min=-1,
    #         b_max=1,
    #         clip=True,
    #     ),
    #     ToTensord(keys=["image"]),
    # ])

In [1]:
from collections.abc import Callable
from functools import partial

from monai.networks.layers.factories import Conv, Pool
from monai.networks.layers.utils import get_act_layer, get_norm_layer, get_pool_layer
from monai.utils import ensure_tuple_rep
from monai.utils.module import look_up_option
from monai.networks.nets.resnet import ResNetBlock, ResNetBottleneck, get_avgpool

import torch
import torch.nn as nn

class ResNet(nn.Module):
    """
    ResNet based on: `Deep Residual Learning for Image Recognition <https://arxiv.org/pdf/1512.03385.pdf>`_
    and `Can Spatiotemporal 3D CNNs Retrace the History of 2D CNNs and ImageNet? <https://arxiv.org/pdf/1711.09577.pdf>`_.
    Adapted from `<https://github.com/kenshohara/3D-ResNets-PyTorch/tree/master/models>`_.

    Args:
        block: which ResNet block to use, either Basic or Bottleneck.
            ResNet block class or str.
            for Basic: ResNetBlock or 'basic'
            for Bottleneck: ResNetBottleneck or 'bottleneck'
        layers: how many layers to use.
        block_inplanes: determine the size of planes at each step. Also tunable with widen_factor.
        spatial_dims: number of spatial dimensions of the input image.
        n_input_channels: number of input channels for first convolutional layer.
        conv1_t_size: size of first convolution layer, determines kernel and padding.
        conv1_t_stride: stride of first convolution layer.
        no_max_pool: bool argument to determine if to use maxpool layer.
        shortcut_type: which downsample block to use. Options are 'A', 'B', default to 'B'.
            - 'A': using `self._downsample_basic_block`.
            - 'B': kernel_size 1 conv + norm.
        widen_factor: widen output for each layer.
        num_classes: number of output (classifications).
        feed_forward: whether to add the FC layer for the output, default to `True`.
        bias_downsample: whether to use bias term in the downsampling block when `shortcut_type` is 'B', default to `True`.
        act: activation type and arguments. Defaults to relu.
        norm: feature normalization type and arguments. Defaults to batch norm.

    """

    def __init__(
        self,
        block: type[ResNetBlock | ResNetBottleneck] | str,
        layers: list[int],
        block_inplanes: list[int],
        spatial_dims: int = 3,
        n_input_channels: int = 3,
        conv1_t_size: tuple[int] | int = 7,
        conv1_t_stride: tuple[int] | int = 1,
        no_max_pool: bool = False,
        shortcut_type: str = "B",
        widen_factor: float = 1.0,
        num_classes: int = 400,
        feed_forward: bool = True,
        bias_downsample: bool = True,  # for backwards compatibility (also see PR #5477)
        act: str | tuple = ("relu", {"inplace": True}),
        norm: str | tuple = "batch",
    ) -> None:
        super().__init__()

        if isinstance(block, str):
            if block == "basic":
                block = ResNetBlock
            elif block == "bottleneck":
                block = ResNetBottleneck
            else:
                raise ValueError("Unknown block '%s', use basic or bottleneck" % block)

        conv_type: type[nn.Conv1d | nn.Conv2d | nn.Conv3d] = Conv[Conv.CONV, spatial_dims]
        pool_type: type[nn.MaxPool1d | nn.MaxPool2d | nn.MaxPool3d] = Pool[Pool.MAX, spatial_dims]
        avgp_type: type[nn.AdaptiveAvgPool1d | nn.AdaptiveAvgPool2d | nn.AdaptiveAvgPool3d] = Pool[
            Pool.ADAPTIVEAVG, spatial_dims
        ]

        block_avgpool = get_avgpool()
        block_inplanes = [int(x * widen_factor) for x in block_inplanes]

        self.in_planes = block_inplanes[0]
        self.no_max_pool = no_max_pool
        self.bias_downsample = bias_downsample

        conv1_kernel_size = ensure_tuple_rep(conv1_t_size, spatial_dims)
        conv1_stride = ensure_tuple_rep(conv1_t_stride, spatial_dims)

        self.conv1 = conv_type(
            n_input_channels,
            self.in_planes,
            kernel_size=conv1_kernel_size,
            stride=conv1_stride,
            padding=tuple(k // 2 for k in conv1_kernel_size),
            bias=False,
        )

        norm_layer = get_norm_layer(name=norm, spatial_dims=spatial_dims, channels=self.in_planes)
        self.bn1 = norm_layer
        self.act = get_act_layer(name=act)
        self.maxpool = pool_type(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, block_inplanes[0], layers[0], spatial_dims, shortcut_type)
        self.layer2 = self._make_layer(block, block_inplanes[1], layers[1], spatial_dims, shortcut_type, stride=2)
        self.layer3 = self._make_layer(block, block_inplanes[2], layers[2], spatial_dims, shortcut_type, stride=2)
        self.layer4 = self._make_layer(block, block_inplanes[3], layers[3], spatial_dims, shortcut_type, stride=2)
        self.avgpool = avgp_type(block_avgpool[spatial_dims])
        self.fc = nn.Linear(block_inplanes[3] * block.expansion, num_classes) if feed_forward else None

        for m in self.modules():
            if isinstance(m, conv_type):
                nn.init.kaiming_normal_(torch.as_tensor(m.weight), mode="fan_out", nonlinearity="relu")
            elif isinstance(m, type(norm_layer)):
                nn.init.constant_(torch.as_tensor(m.weight), 1)
                nn.init.constant_(torch.as_tensor(m.bias), 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(torch.as_tensor(m.bias), 0)

    def _downsample_basic_block(self, x: torch.Tensor, planes: int, stride: int, spatial_dims: int = 3) -> torch.Tensor:
        out: torch.Tensor = get_pool_layer(("avg", {"kernel_size": 1, "stride": stride}), spatial_dims=spatial_dims)(x)
        zero_pads = torch.zeros(out.size(0), planes - out.size(1), *out.shape[2:], dtype=out.dtype, device=out.device)
        out = torch.cat([out.data, zero_pads], dim=1)
        return out

    def _make_layer(
        self,
        block: type[ResNetBlock | ResNetBottleneck],
        planes: int,
        blocks: int,
        spatial_dims: int,
        shortcut_type: str,
        stride: int = 1,
        norm: str | tuple = "batch",
    ) -> nn.Sequential:
        conv_type: Callable = Conv[Conv.CONV, spatial_dims]

        downsample: nn.Module | partial | None = None
        if stride != 1 or self.in_planes != planes * block.expansion:
            if look_up_option(shortcut_type, {"A", "B"}) == "A":
                downsample = partial(
                    self._downsample_basic_block,
                    planes=planes * block.expansion,
                    stride=stride,
                    spatial_dims=spatial_dims,
                )
            else:
                downsample = nn.Sequential(
                    conv_type(
                        self.in_planes,
                        planes * block.expansion,
                        kernel_size=1,
                        stride=stride,
                        bias=self.bias_downsample,
                    ),
                    get_norm_layer(name=norm, spatial_dims=spatial_dims, channels=planes * block.expansion),
                )

        layers = [
            block(
                in_planes=self.in_planes,
                planes=planes,
                spatial_dims=spatial_dims,
                stride=stride,
                downsample=downsample,
                norm=norm,
            )
        ]

        self.in_planes = planes * block.expansion
        for _i in range(1, blocks):
            layers.append(block(self.in_planes, planes, spatial_dims=spatial_dims, norm=norm))

        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act(x)
        if not self.no_max_pool:
            x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # x = self.avgpool(x)

        # x = x.view(x.size(0), -1)
        # if self.fc is not None:
        #     x = self.fc(x)

        return x

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

model = ResNet(
    block="basic",           # Use Basic block for 18/34
    layers=[2, 2, 2, 2],         # Standard ResNet-18 configuration
    block_inplanes=[16, 32, 64, 128],
    spatial_dims=3,              # Set to 3 for 3D medical volumes
    n_input_channels=1,          # Single channel (e.g., CT scan)
    num_classes=None,                # Binary classification
    feed_forward=False,
    no_max_pool=False,
)

In [3]:
import torch
device = torch.device("cuda")
model.train()
model.to(device)

ResNet(
  (conv1): Conv3d(1, 16, kernel_size=(7, 7, 7), stride=(1, 1, 1), padding=(3, 3, 3), bias=False)
  (bn1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act): ReLU(inplace=True)
  (maxpool): MaxPool3d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): ResNetBlock(
      (conv1): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): ReLU(inplace=True)
      (conv2): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn2): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResNetBlock(
      (conv1): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [ ]:
import torch.nn.functional as F

x = torch.rand(8,1,256,256,100, device=device, dtype=torch.float32)
o = model(x)
print(o.shape)
avg_output = F.adaptive_avg_pool3d(o, (1,1,1))
print(avg_output.shape)

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.56 GiB. GPU 0 has a total capacity of 14.83 GiB of which 1.00 GiB is free. Of the allocated memory 13.68 GiB is allocated by PyTorch, and 16.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)